In [2]:
import os
import json
import pandas as pd

DATA_DIR = "data"

valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

records = []
for idx, cname in enumerate(class_names):
    cpath = os.path.join(DATA_DIR, cname)
    for fname in os.listdir(cpath):
        if fname.lower().endswith(valid_exts):
            records.append({
                "filepath": os.path.join(cpath, fname),
                "label": idx,
                "class_name": cname
            })

df = pd.DataFrame(records)
print("Total classes:", len(class_names))
print("Total images:", len(df))
print("\nSample classes:", class_names[:5])

with open("class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)

Total classes: 38
Total images: 54306

Sample classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [4]:
from sklearn.model_selection import train_test_split

# 1. Pehle 15% test set alag karein (final evaluation ke liye)
train_pool_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

# 2. Train pool mein se 20% Labeled aur 80% Unlabeled banayein
labeled_df, unlabeled_df = train_test_split(
    train_pool_df,
    train_size=0.20,
    stratify=train_pool_df["label"],
    random_state=42
)

# Unlabeled set mein labels hide/drop kar dete hain (pseudo-labeling ke liye)
unlabeled_df = unlabeled_df[["filepath"]].copy()

print("Labeled train images:", len(labeled_df))
print("Unlabeled images:    ", len(unlabeled_df))
print("Test images:         ", len(test_df))

Labeled train images: 9232
Unlabeled images:     36928
Test images:          8146


In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
NUM_CLASSES = len(class_names)  # 38
EPOCHS_TEACHER = 5  # Pehle fast check ke liye 5 epochs; baad mein 10-15 kar sakte hain

def decode_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32)

def make_labeled_ds(df_sub, training=True, shuffle=True):
    paths = df_sub["filepath"].values
    labels = df_sub["label"].values.astype(np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=42, reshuffle_each_iteration=True)

    augmenter = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
    ])

    def _load(p, y):
        x = decode_image(p)
        if training:
            x = augmenter(x, training=True)
        x = preprocess_input(x)
        return x, y

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def make_unlabeled_ds(filepaths):
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    def _load_u(p):
        x = decode_image(p)
        x = preprocess_input(x)
        return x
    ds = ds.map(_load_u, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# Train aur Validation split labeled_df ke andar se (90% train / 10% val)
tr_sub, val_sub = train_test_split(
    labeled_df, test_size=0.1, stratify=labeled_df["label"], random_state=42
)

train_labeled_ds = make_labeled_ds(tr_sub, training=True, shuffle=True)
val_labeled_ds   = make_labeled_ds(val_sub, training=False, shuffle=False)
test_ds          = make_labeled_ds(test_df, training=False, shuffle=False)

def build_cnn(num_classes):
    base = MobileNetV2(
        input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
        include_top=False,
        weights="imagenet"
    )
    base.trainable = False  # Pehle top classifier head train karenge

    inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inputs, outputs)

teacher_model = build_cnn(NUM_CLASSES)
teacher_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_teacher = teacher_model.fit(
    train_labeled_ds,
    validation_data=val_labeled_ds,
    epochs=EPOCHS_TEACHER
)

test_loss, test_acc = teacher_model.evaluate(test_ds, verbose=0)
print(f"\nTeacher Test Accuracy (20% labeled data only): {test_acc:.4f}")

Epoch 1/5


C:\Users\Hk\AppData\Roaming\Python\Python314\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


260/260 ━━━━━━━━━━━━━━━━━━━━ 61s 221ms/step - accuracy: 0.7047 - loss: 1.1316 - val_accuracy: 0.8506 - val_loss: 0.5970
Epoch 2/5
260/260 ━━━━━━━━━━━━━━━━━━━━ 58s 223ms/step - accuracy: 0.8660 - loss: 0.4697 - val_accuracy: 0.8647 - val_loss: 0.4868
Epoch 3/5
260/260 ━━━━━━━━━━━━━━━━━━━━ 60s 228ms/step - accuracy: 0.8965 - loss: 0.3559 - val_accuracy: 0.8810 - val_loss: 0.4140
Epoch 4/5
260/260 ━━━━━━━━━━━━━━━━━━━━ 60s 229ms/step - accuracy: 0.9011 - loss: 0.3131 - val_accuracy: 0.8799 - val_loss: 0.3919
Epoch 5/5
260/260 ━━━━━━━━━━━━━━━━━━━━ 63s 241ms/step - accuracy: 0.9214 - loss: 0.2696 - val_accuracy: 0.8874 - val_loss: 0.3616

Teacher Test Accuracy (20% labeled data only): 0.8879


In [7]:
CONF_THRESHOLD = 0.90
EPOCHS_STUDENT = 5

# 1. Unlabeled images par Teacher ki predictions
unlabeled_paths = unlabeled_df["filepath"].values
unlabeled_ds = make_unlabeled_ds(unlabeled_paths)

probs = teacher_model.predict(unlabeled_ds, verbose=1)
pred_labels = np.argmax(probs, axis=1)
max_probs = np.max(probs, axis=1)

# 2. Sirf >= 0.90 confidence wale samples select karein
mask = max_probs >= CONF_THRESHOLD
pseudo_df = pd.DataFrame({
    "filepath": unlabeled_paths[mask],
    "label": pred_labels[mask].astype(np.int32)
})

print(f"\nTotal unlabeled images: {len(unlabeled_paths)}")
print(f"Pseudo-labeled (>= {CONF_THRESHOLD}): {len(pseudo_df)} ({len(pseudo_df)/len(unlabeled_paths)*100:.2f}%)")

# 3. Labeled train + Pseudo-labeled data ko combine karein
combined_train_df = pd.concat(
    [tr_sub[["filepath", "label"]], pseudo_df],
    ignore_index=True
)
print("Combined Student training images:", len(combined_train_df))

train_student_ds = make_labeled_ds(combined_train_df, training=True, shuffle=True)

# 4. Student CNN build, train, evaluate aur save karein
student_model = build_cnn(NUM_CLASSES)
student_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_student = student_model.fit(
    train_student_ds,
    validation_data=val_labeled_ds,
    epochs=EPOCHS_STUDENT
)

student_loss, student_acc = student_model.evaluate(test_ds, verbose=0)
print(f"\nTeacher Test Accuracy: {test_acc:.4f}")
print(f"Student Test Accuracy: {student_acc:.4f}")

student_model.save("ssl_plant_disease_cnn.keras")
print("Saved model to ssl_plant_disease_cnn.keras")

1154/1154 ━━━━━━━━━━━━━━━━━━━━ 257s 221ms/step

Total unlabeled images: 36928
Pseudo-labeled (>= 0.9): 25060 (67.86%)
Combined Student training images: 33368
Epoch 1/5


C:\Users\Hk\AppData\Roaming\Python\Python314\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1043/1043 ━━━━━━━━━━━━━━━━━━━━ 232s 219ms/step - accuracy: 0.9041 - loss: 0.3847 - val_accuracy: 0.8626 - val_loss: 0.4733
Epoch 2/5
1043/1043 ━━━━━━━━━━━━━━━━━━━━ 248s 238ms/step - accuracy: 0.9586 - loss: 0.1422 - val_accuracy: 0.8680 - val_loss: 0.4592
Epoch 3/5
1043/1043 ━━━━━━━━━━━━━━━━━━━━ 226s 216ms/step - accuracy: 0.9656 - loss: 0.1161 - val_accuracy: 0.8842 - val_loss: 0.4098
Epoch 4/5
1043/1043 ━━━━━━━━━━━━━━━━━━━━ 227s 217ms/step - accuracy: 0.9672 - loss: 0.1047 - val_accuracy: 0.8810 - val_loss: 0.3902
Epoch 5/5
1043/1043 ━━━━━━━━━━━━━━━━━━━━ 227s 217ms/step - accuracy: 0.9705 - loss: 0.0934 - val_accuracy: 0.8669 - val_loss: 0.4656

Teacher Test Accuracy: 0.8879
Student Test Accuracy: 0.8673
Saved model to ssl_plant_disease_cnn.keras
